# 01_exploration.ipynb

Exploratory Data Analysis (EDA) for Practical Foundations for Question Intelligence

This notebook explores the raw synthetic data in `data/raw/`:
- Learner questions (`learner_questions.csv`)
- Lessons (`lessons.csv`)
- Users (`users.csv`)

We examine data quality, distributions, and initial hypotheses to inform downstream cleaning and modeling.

In [ ]:
# Set up project imports and paths
import sys
from pathlib import Path

# Add project root to sys.path for src imports
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# Import project data loaders
from src.data.load import load_questions, load_lessons, load_users

## 1. Load Raw Data

In [ ]:
# Load raw data using project loaders
questions_raw = load_questions()
lessons_raw = load_lessons()
users_raw = load_users()

print(f"Questions: {questions_raw.shape}")
print(f"Lessons: {lessons_raw.shape}")
print(f"Users: {users_raw.shape}")

## 2. Inspect Learner Questions

In [ ]:
# Show sample rows
questions_raw.head(8)

In [ ]:
# Column info and missing values
questions_raw.info()
questions_raw.isna().sum()

In [ ]:
# Check for empty or whitespace-only texts
empty_texts = questions_raw['text'].apply(lambda x: not str(x).strip())
print(f"Empty/blank question texts: {empty_texts.sum()} out of {len(questions_raw)}")

In [ ]:
# Check for duplicate question texts
dup_texts = questions_raw.duplicated(subset=['text'])
print(f"Questions with duplicate text: {dup_texts.sum()}")

In [ ]:
# Distribution of labels (if present)
if 'label' in questions_raw.columns:
    print(questions_raw['label'].value_counts(dropna=False))

In [ ]:
# Timestamp format examples
questions_raw['timestamp'].sample(5, random_state=1)

In [ ]:
# Parse timestamps to datetime (show parse errors)
def try_parse(ts):
    try:
        return pd.to_datetime(ts)
    except Exception:
        return np.nan
parsed = questions_raw['timestamp'].apply(try_parse)
print(f"Unparseable timestamps: {(parsed.isna()).sum()}")

### Question Length Distribution

In [ ]:
questions_raw['text_len'] = questions_raw['text'].astype(str).apply(len)
plt.figure(figsize=(8, 4))
sns.histplot(questions_raw['text_len'], bins=20, kde=True)
plt.title('Distribution of Question Text Lengths')
plt.xlabel('Number of Characters')
plt.ylabel('Count')
plt.show()

### Questions per Lesson

In [ ]:
q_per_lesson = questions_raw.groupby('lesson_id').size().sort_values(ascending=False)
plt.figure(figsize=(10, 4))
sns.barplot(x=q_per_lesson.index, y=q_per_lesson.values, color='skyblue')
plt.title('Number of Questions per Lesson')
plt.xlabel('Lesson ID')
plt.ylabel('Number of Questions')
plt.xticks(rotation=45)
plt.show()

### Questions per User

In [ ]:
q_per_user = questions_raw.groupby('user_id').size().sort_values(ascending=False)
plt.figure(figsize=(10, 4))
sns.histplot(q_per_user, bins=15, kde=False)
plt.title('Distribution of Questions per User')
plt.xlabel('Questions per User')
plt.ylabel('Number of Users')
plt.show()

## 3. Inspect Lessons

In [ ]:
lessons_raw.head(5)

In [ ]:
lessons_raw.info()
lessons_raw.isna().sum()

In [ ]:
# Check for duplicate titles
dup_titles = lessons_raw.duplicated(subset=['title'])
print(f"Lessons with duplicate titles: {dup_titles.sum()}")

In [ ]:
# Missing topics
print(f"Lessons with missing topic: {(lessons_raw['topic'].astype(str).str.strip() == '').sum()}")

### Lessons by Topic

In [ ]:
topic_counts = lessons_raw['topic'].astype(str).str.strip().value_counts(dropna=False)
topic_counts.plot(kind='bar', figsize=(8, 4), color='coral')
plt.title('Lesson Counts by Topic')
plt.xlabel('Topic')
plt.ylabel('Number of Lessons')
plt.show()

## 4. Inspect Users

In [ ]:
users_raw.head(5)

In [ ]:
users_raw.info()
users_raw.isna().sum()

In [ ]:
# Duplicate user IDs
dup_users = users_raw.duplicated(subset=['id'])
print(f"Duplicate user IDs: {dup_users.sum()}")

In [ ]:
# Missing cohort
print(f"Users with missing cohort: {(users_raw['cohort'].astype(str).str.strip() == '').sum()}")

### Users by Role

In [ ]:
role_counts = users_raw['role'].astype(str).str.strip().value_counts(dropna=False)
role_counts.plot(kind='bar', figsize=(6, 3), color='mediumseagreen')
plt.title('User Counts by Role')
plt.xlabel('Role')
plt.ylabel('Number of Users')
plt.show()

## 5. Cross-Referencing: Data Integrity Checks

In [ ]:
# Are all question user_ids present in users?
missing_users = set(questions_raw['user_id']) - set(users_raw['id'])
print(f"Questions with user_id not in users.csv: {len(missing_users)}")

In [ ]:
# Are all question lesson_ids present in lessons?
missing_lessons = set(questions_raw['lesson_id']) - set(lessons_raw['id'])
print(f"Questions with lesson_id not in lessons.csv: {len(missing_lessons)}")

## 6. Initial Observations & Hypotheses

- There are missing and duplicate values in all three datasets, as expected from the synthetic data spec.
- Some question texts are empty or duplicated; timestamps are inconsistently formatted.
- Some lessons lack topics or have duplicate titles.
- Some users have duplicate IDs or missing cohort values.
- Not all question user_ids/lesson_ids are present in users/lessons, indicating referential integrity issues.

### Next Steps
- Clean and preprocess data using `src/data/clean.py`.
- Explore text features and clustering in the next notebook.
- Document data quality issues for model limitations reporting.